# YOLOv1 — Training on Google Colab
**Paper:** *You Only Look Once: Unified, Real-Time Object Detection* (CVPR 2016)

This notebook uploads the project zip, extracts it, and trains YOLOv1 on Colab's free GPU.

**Important:** Go to `Runtime → Change runtime type → GPU` before running.

## 1. Setup — Upload Zip & Extract

In [ ]:
import os, sys, shutil, glob, torch

# Auto-find project files in Kaggle input
py_files = glob.glob("/kaggle/input/**/*.py", recursive=True)
if not py_files:
    raise FileNotFoundError("No .py files found — check your dataset is added!")

PROJECT_INPUT = os.path.dirname(py_files[0])
print(f"Found project at: {PROJECT_INPUT}")

WORK_DIR = "/kaggle/working"

# Copy .py files to working dir (input folder is read-only on Kaggle)
for f in glob.glob(f"{PROJECT_INPUT}/*.py"):
    shutil.copy(f, WORK_DIR)

# Symlink data folder so config.py finds it at /kaggle/working/data
if not os.path.exists(f"{WORK_DIR}/data"):
    os.symlink(f"{PROJECT_INPUT}/data", f"{WORK_DIR}/data")

os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

print(f"Working dir: {os.getcwd()}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU found! Go to Settings → Accelerator → GPU")


In [ ]:
print(f"Train: {len(os.listdir('data/train/images'))} images")
print(f"Valid: {len(os.listdir('data/valid/images'))} images")
print(f"Test:  {len(os.listdir('data/test/images'))} images")


In [ ]:
# Verify PyTorch, GPU, and add project to Python path
import sys
import os
import torch

# Ensure /content/yolov1 is on the Python path so all our modules are importable
PROJECT_DIR = "/content/yolov1"
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print(f"Working directory: {os.getcwd()}")
print(f"Python path includes: {PROJECT_DIR}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"\nProject files: {os.listdir(PROJECT_DIR)}")

## 2. Verify Architecture

In [ ]:
from model import YOLOv1
from config import NUM_CLASSES, B, S

model = YOLOv1(num_classes=NUM_CLASSES, num_boxes=B, grid_size=S)
total_params = sum(p.numel() for p in model.parameters())
print(f"YOLOv1 — Total parameters: {total_params:,}")

# Sanity check forward pass
dummy = torch.randn(1, 3, 448, 448)
out = model(dummy)
print(f"Input:  {dummy.shape}")
print(f"Output: {out.shape}")
assert out.shape == (1, 7, 7, 14), f"Expected (1,7,7,14), got {out.shape}"
print("Architecture OK")

## 3. Visualize a Training Sample

In [ ]:
from dataset import YOLODataset
from config import TRAIN_IMG_DIR, TRAIN_LABEL_DIR
from visualize import visualize_sample
from IPython.display import display
from PIL import Image

ds = YOLODataset(TRAIN_IMG_DIR, TRAIN_LABEL_DIR, augment=False)
print(f"Dataset size: {len(ds)}")

img, target = ds[0]
print(f"Image shape: {img.shape}, Target shape: {target.shape}")
n_obj = (target[:, :, 4] > 0).sum().item()
print(f"Objects in first sample: {n_obj}")

# Visualize with grid and GT boxes
vis_img = visualize_sample(ds, 0, "sample_vis.jpg")
display(vis_img)

## 4. Train YOLOv1

In [ ]:
import os
import time
import torch
from torch.utils.data import DataLoader

from config import (
    TRAIN_IMG_DIR, TRAIN_LABEL_DIR,
    VALID_IMG_DIR, VALID_LABEL_DIR,
    BATCH_SIZE, MOMENTUM, WEIGHT_DECAY, NUM_EPOCHS,
    LR_WARMUP_EPOCHS, LR_INIT, LR_MAIN, LR_SCHEDULE,
    CHECKPOINT_DIR, S, B, NUM_CLASSES,
)
from model import YOLOv1
from loss import YOLOv1Loss
from dataset import YOLODataset
from train import get_lr, train_one_epoch, validate

# ── Device ──
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

# ── Datasets ──
train_dataset = YOLODataset(TRAIN_IMG_DIR, TRAIN_LABEL_DIR, augment=True)
valid_dataset = YOLODataset(VALID_IMG_DIR, VALID_LABEL_DIR, augment=False)
print(f"Train: {len(train_dataset)}, Valid: {len(valid_dataset)}")

effective_batch = min(BATCH_SIZE, len(train_dataset))

train_loader = DataLoader(
    train_dataset, batch_size=effective_batch, shuffle=True,
    num_workers=2, pin_memory=True, drop_last=True,
)
valid_loader = DataLoader(
    valid_dataset, batch_size=effective_batch, shuffle=False,
    num_workers=2, pin_memory=True,
)

# ── Model ──
model = YOLOv1(num_classes=NUM_CLASSES, num_boxes=B, grid_size=S).to(device)
criterion = YOLOv1Loss()

# ── Optimizer (SGD — as per paper Section 2.2) ──
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=LR_INIT,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ── Training Loop ──
best_mAP = 0.0
train_losses = []
val_losses = []
mAP_history = []

for epoch in range(NUM_EPOCHS):
    t_start = time.time()

    # Update learning rate (manual schedule from the paper)
    lr = get_lr(epoch)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    # Train one epoch
    train_loss = train_one_epoch(model, train_loader, criterion,
                                 optimizer, device, epoch)
    train_losses.append(train_loss)

    # Validate every 5 epochs
    val_loss, mAP, per_class_ap = 0.0, 0.0, {}
    if (epoch + 1) % 5 == 0 or epoch == NUM_EPOCHS - 1:
        val_loss, mAP, per_class_ap = validate(
            model, valid_loader, criterion, device
        )
        val_losses.append(val_loss)
        mAP_history.append((epoch + 1, mAP))

    elapsed = time.time() - t_start

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}]  LR: {lr:.6f}  "
          f"Train Loss: {train_loss:.4f}  Val Loss: {val_loss:.4f}  "
          f"mAP: {mAP:.4f}  Time: {elapsed:.1f}s")

    if per_class_ap:
        for cls_name, ap in per_class_ap.items():
            print(f"  {cls_name}: AP={ap:.4f}")

    # Save best model
    if mAP > best_mAP:
        best_mAP = mAP
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'mAP': mAP,
            'train_loss': train_loss,
        }, os.path.join(CHECKPOINT_DIR, "yolov1_best.pth"))
        print(f"  >> Saved best model (mAP={mAP:.4f})")

    # Save every 25 epochs
    if (epoch + 1) % 25 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'mAP': mAP,
        }, os.path.join(CHECKPOINT_DIR, f"yolov1_epoch{epoch+1}.pth"))

# Save final
torch.save({
    'epoch': NUM_EPOCHS - 1,
    'model_state_dict': model.state_dict(),
    'mAP': mAP,
}, os.path.join(CHECKPOINT_DIR, "yolov1_final.pth"))

print(f"\nTraining complete! Best mAP: {best_mAP:.4f}")

## 5. Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
ax1.plot(range(1, len(train_losses) + 1), train_losses, label='Train Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.legend()
ax1.grid(True)

# mAP curve
if mAP_history:
    epochs_eval, maps = zip(*mAP_history)
    ax2.plot(epochs_eval, maps, 'o-', color='green', label='mAP@0.5')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('mAP')
    ax2.set_title('Validation mAP@0.5')
    ax2.legend()
    ax2.grid(True)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## 6. Evaluate on Validation Set

In [ ]:
from evaluate import evaluate as full_evaluate
from config import CONF_THRESHOLD, NMS_THRESHOLD

# Load best checkpoint
ckpt = torch.load(os.path.join(CHECKPOINT_DIR, "yolov1_best.pth"), map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded best checkpoint from epoch {ckpt['epoch'] + 1}")

val_loss, mAP, per_class_ap = full_evaluate(
    model, valid_loader, criterion, device,
    conf_thresh=CONF_THRESHOLD, nms_thresh=NMS_THRESHOLD
)

print(f"\nValidation Results:")
print(f"  Loss:    {val_loss:.4f}")
print(f"  mAP@0.5: {mAP:.4f}")
print(f"\nPer-class AP:")
for cls_name, ap in per_class_ap.items():
    print(f"  {cls_name:10s}: {ap:.4f}")

## 7. Run Detection on Test Images

In [ ]:
from detect import detect_single
from config import TEST_IMG_DIR
from IPython.display import display
import glob

# Get some test images
test_images = sorted(glob.glob(os.path.join(TEST_IMG_DIR, "*.jpg")))[:6]
print(f"Showing detections on {len(test_images)} test images\n")

model.eval()

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, img_path in enumerate(test_images):
    dets, result_img = detect_single(model, img_path, device)
    n_dets = dets.size(0) if dets.numel() > 0 else 0

    axes[idx].imshow(result_img)
    axes[idx].set_title(f"{os.path.basename(img_path)[:30]}... ({n_dets} dets)")
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('test_detections.png', dpi=150)
plt.show()

## 8. Download Best Model

In [ ]:
from google.colab import files

# Download the best checkpoint
best_path = os.path.join(CHECKPOINT_DIR, "yolov1_best.pth")
if os.path.exists(best_path):
    files.download(best_path)
    print("Downloading best model checkpoint...")
else:
    print("No best checkpoint found. Train the model first.")